# Question-Level Survey Analysis

This notebook analyzes each survey question (`q_*`) one by one.
It shows how answers vary by gender, age, education, probashi status,
economic and social scores, and political alignment (`resultLabel`).

If you later add a question-to-statement mapping, replace the values in
`question_labels` so the notebook can show readable statement text.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


In [2]:
df = pd.read_parquet("data/responses.parquet")

question_columns = sorted(
    [column_name for column_name in df.columns if column_name.startswith("q_")],
    key=lambda column_name: int(column_name.split("_")[1]),
)

question_labels = {}
questions_file = Path("questions_v1.parquet")

if questions_file.exists():
    questions_df = pd.read_parquet(questions_file)
    question_labels = {
        f"q_{int(question_id)}": question_text
        for question_id, question_text in zip(questions_df["id"], questions_df["text"])
        if pd.notna(question_id) and pd.notna(question_text)
    }
    print(f"Loaded {len(question_labels)} question labels from questions_v1.parquet")
else:
    print("questions_v1.parquet not found. Falling back to question IDs.")

print(f"Total responses: {len(df)}")
print(f"Questions found: {len(question_columns)}")
print(question_columns)


Loaded 26 question labels from questions_v1.parquet
Total responses: 17259
Questions found: 27
['q_1', 'q_2', 'q_3', 'q_4', 'q_5', 'q_6', 'q_7', 'q_8', 'q_9', 'q_10', 'q_11', 'q_12', 'q_13', 'q_14', 'q_15', 'q_16', 'q_17', 'q_18', 'q_19', 'q_20', 'q_21', 'q_22', 'q_101', 'q_102', 'q_103', 'q_104', 'q_105']


In [3]:
def add_missing_label(series):
    """Convert missing values into a readable label for tables."""
    return series.astype("object").where(series.notna(), "Missing")


def percentage_table(group_series, answer_series):
    """Show how each group answered a question in percentage terms."""
    table = pd.crosstab(
        add_missing_label(group_series),
        add_missing_label(answer_series),
        normalize="index",
    )
    return table.mul(100).round(1)


def overall_answer_percentages(answer_series):
    """Show the overall answer split for a question."""
    return answer_series.value_counts(normalize=True).sort_index().mul(100).round(1)


def score_by_answer(question_frame, question_column):
    """Show average economic and social scores for each answer option."""
    table = (
        question_frame.groupby(question_column)[["economicScore", "socialScore"]]
        .mean()
        .round(2)
        .sort_index()
    )
    return table


def analyze_question(question_column):
    """Display a compact analysis block for one survey question."""
    question_title = question_labels.get(question_column, question_column)
    question_frame = df[df[question_column].notna()].copy()

    display(Markdown(f"## {question_title}"))
    print(f"Non-null responses: {len(question_frame)}")

    print("\nOverall answer percentages:")
    print(overall_answer_percentages(question_frame[question_column]))

    print("\nAnswer percentages by gender:")
    print(percentage_table(question_frame["gender"], question_frame[question_column]))

    print("\nAnswer percentages by age:")
    print(percentage_table(question_frame["age"], question_frame[question_column]))

    print("\nAnswer percentages by education:")
    print(percentage_table(question_frame["education"], question_frame[question_column]))

    print("\nAnswer percentages by probashi status:")
    print(percentage_table(question_frame["probashi"], question_frame[question_column]))

    print("\nAverage economic and social scores by answer:")
    print(score_by_answer(question_frame, question_column))

    print("\nAnswer percentages by political alignment:")
    print(percentage_table(question_frame["resultLabel"], question_frame[question_column]))

    print("\n" + "-" * 80 + "\n")


In [4]:
for question_column in question_columns:
    analyze_question(question_column)


## দেশের বড় বড় কল-কারখানা ও ব্যবসা সরকারের অধীনে থাকা উচিত।

Non-null responses: 17259

Overall answer percentages:
q_1
-2    21.6
-1    39.9
 0     8.9
 1    21.5
 2     8.2
Name: proportion, dtype: float64

Answer percentages by gender:
q_1       -2    -1     0     1     2
gender                              
female  15.9  38.7  12.4  26.3   6.6
male    22.3  40.3   8.2  21.0   8.2
other   20.6  17.0  30.9  12.1  19.4

Answer percentages by age:
q_1      -2    -1     0     1     2
age                                
18-24  20.4  38.7  10.1  22.7   8.1
25-34  23.1  41.6   7.2  20.3   7.8
35-44  22.6  43.4   6.0  17.9  10.2
45-54  28.0  39.2   4.9  16.1  11.9
55+    27.6  22.4  15.8   9.2  25.0

Answer percentages by education:
q_1                 -2    -1     0     1     2
education                                     
bachelor_masters  21.3  41.5   8.3  21.5   7.5
below_ssc         33.9  22.9  10.0  18.6  14.6
phd               28.1  27.5   7.7  15.7  20.9
ssc_hsc           20.3  38.0  10.8  22.6   8.4

Answer percentages by probashi status:
q

## ব্যবসা-বাণিজ্য করার সুযোগ সবার জন্য উন্মুক্ত থাকা উচিত, সরকার এতে বাধা দেবে না।

Non-null responses: 17259

Overall answer percentages:
q_2
-2     4.3
-1     7.0
 0     2.9
 1    46.7
 2    39.1
Name: proportion, dtype: float64

Answer percentages by gender:
q_2      -2    -1     0     1     2
gender                             
female  3.6  11.3   3.9  51.0  30.1
male    4.4   6.5   2.5  46.5  40.1
other   9.7  11.5  26.1  18.2  34.5

Answer percentages by age:
q_2      -2   -1    0     1     2
age                              
18-24   4.5  7.3  3.2  45.9  39.1
25-34   3.9  6.8  2.3  48.0  39.0
35-44   4.5  5.3  2.4  48.4  39.3
45-54   8.4  3.5  3.5  50.3  34.3
55+    10.5  3.9  7.9  32.9  44.7

Answer percentages by education:
q_2                 -2   -1    0     1     2
education                                   
bachelor_masters   3.4  7.0  2.7  47.6  39.3
below_ssc         21.2  6.6  4.8  35.6  31.7
phd               12.9  9.1  2.2  30.6  45.2
ssc_hsc            4.2  7.0  3.2  47.0  38.6

Answer percentages by probashi status:
q_2        -2   -1    0     1   

## ধনীদের কাছ থেকে বেশি ট্যাক্স নিয়ে গরিবদের সাহায্য করা উচিত।

...

Non-null responses: 17259

Overall answer percentages:
q_103
business      5.2
homemaker     0.8
other         7.0
service      18.0
student      69.0
Name: proportion, dtype: float64

Answer percentages by gender:
q_103   business  homemaker  other  service  student
gender                                              
female       1.6        5.0    6.9     12.7     73.9
male         5.4        0.2    6.8     18.7     68.8
other       18.2       10.3   27.3     12.1     32.1

Answer percentages by age:
q_103  business  homemaker  other  service  student
age                                                
18-24       1.7        0.3    2.3      3.7     92.0
25-34       8.4        0.8   12.6     35.7     42.5
35-44      20.1        3.3   17.2     54.2      5.2
45-54      23.1       14.0   16.8     37.8      8.4
55+        15.8        9.2   44.7     17.1     13.2

Answer percentages by education:
q_103             business  homemaker  other  service  student
education                      

## আপনি কি প্রবাসী? (Are you an Expatriate?)

Non-null responses: 17259

Overall answer percentages:
q_104
no     92.6
yes     7.4
Name: proportion, dtype: float64

Answer percentages by gender:
q_104     no   yes
gender            
female  95.1   4.9
male    92.5   7.5
other   79.4  20.6

Answer percentages by age:
q_104    no   yes
age              
18-24  95.6   4.4
25-34  89.8  10.2
35-44  81.8  18.2
45-54  79.0  21.0
55+    76.3  23.7

Answer percentages by education:
q_104               no   yes
education                   
bachelor_masters  94.1   5.9
below_ssc         77.5  22.5
phd               65.6  34.4
ssc_hsc           92.5   7.5

Answer percentages by probashi status:
q_104        no    yes
probashi              
no        100.0    0.0
yes         0.0  100.0

Average economic and social scores by answer:
       economicScore  socialScore
q_104                            
no             -1.07        -0.34
yes            -0.87        -0.58

Answer percentages by political alignment:
q_104              no  yes
resultLa

## q_105

Non-null responses: 17259

Overall answer percentages:
q_105
18-24    57.1
25-34    37.9
35-44     3.7
45-54     0.8
55+       0.4
Name: proportion, dtype: float64

Answer percentages by gender:
q_105   18-24  25-34  35-44  45-54   55+
gender                                  
female   64.8   30.7    2.5    1.6   0.4
male     56.4   38.9    3.7    0.7   0.3
other    41.2   31.5   12.1    3.6  11.5

Answer percentages by age:
q_105  18-24  25-34  35-44  45-54    55+
age                                     
18-24  100.0    0.0    0.0    0.0    0.0
25-34    0.0  100.0    0.0    0.0    0.0
35-44    0.0    0.0  100.0    0.0    0.0
45-54    0.0    0.0    0.0  100.0    0.0
55+      0.0    0.0    0.0    0.0  100.0

Answer percentages by education:
q_105             18-24  25-34  35-44  45-54  55+
education                                        
bachelor_masters   49.4   45.7    3.9    0.8  0.2
below_ssc          76.8   14.0    3.9    2.2  3.1
phd                22.6   45.2   19.6    5.0  7.7
s